# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema URL and contains multiple record sets describing ordered logistic regression outputs, socio-demographics, knowledge management, and intervention outcomes among pastoral households in Samburu, Isiolo, and Marsabit counties, Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print the dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

By inspecting the Croissant schema, we list the available record sets and fields. This ensures all entities are referenced by their `@id`.

In [ ]:
# List available record sets and their fields using @id
record_sets = dataset.metadata.record_set
if not record_sets:
    print("No record sets defined in the metadata.")
else:
    for rs in record_sets:
        print("RecordSet ID:", rs['@id'])
        fields = rs.get('field', [])
        field_ids = [f['@id'] for f in fields]
        print("  Fields:", field_ids)
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s obtained in the overview.

Below, we extract all records for each available record set and load them into pandas DataFrames indexed by record set `@id`.

In [ ]:
# Gather all record set IDs
record_set_ids = []
if dataset.metadata.record_set:
    record_set_ids = [rs['@id'] for rs in dataset.metadata.record_set]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame columns for RecordSet '{record_set_id}':")
        print(df.columns.tolist())
        print(df.head(), '\n')
    else:
        print(f"No records found for record set with ID: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes referenced only by their `@id`.

In [ ]:
# Processing EDA on the first detected record set (if any)
if dataframes:
    # Select the first record set for demonstration
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]

    # Attempt to find numeric fields via @id (e.g., coefficient or log likelihood fields)
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if len(numeric_fields) > 0:
        numeric_field_id = numeric_fields[0]  # Use the first numeric field
        print(f"Using numeric field ID: {numeric_field_id}")

        # Filter for values above threshold and normalize
        threshold = df[numeric_field_id].mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Normalize (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a categorical field
        # Choose first field that is 'object' type and has < 20 unique values
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == 'object' and df[col].nunique() < 20:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean()
            print(f"Grouped data by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in the first record set.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the selected numeric field and a grouped bar chart if a categorical grouping was available.

In [ ]:
# Visualization Example
if dataframes:
    df = dataframes[list(dataframes.keys())[0]]
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    if len(numeric_fields) > 0:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()

        # If a grouping field was used in the previous section...
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == 'object' and df[col].nunique() < 20:
                group_field_id = col
                break
        if group_field_id:
            plt.figure(figsize=(8, 4))
            sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"Mean '{numeric_field_id}' grouped by '{group_field_id}'")
            plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides regression outputs and household characteristics relevant to knowledge adoption in rangeland management.
- By referencing entities using their `@id`, analysis is traceable and adheres to FAIR principles.
- Numeric and categorical fields offer opportunities for filtering, normalization, and grouping.
- Visualizations reveal distribution patterns among regression statistics and potential relationships between predictors and household characteristics.